# Investment Simulation System

In [5]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [6]:
%pip install pandas_ta_classic

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install tqdm 

Note: you may need to restart the kernel to use updated packages.


### Imports

In [ ]:
import os
import site

import os
import shutil
import gc
import warnings
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
from datetime import timedelta

warnings.filterwarnings('ignore')

KeyboardInterrupt: 

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    precision_score, recall_score, f1_score, matthews_corrcoef,
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau


### Configurations

In [ ]:
MODEL_SAVE_PATH = "trained_models/"
MIN_SEQUENCE_LENGTH = 12  # Minimum sequence length for any company
MAX_SEQUENCE_LENGTH = 12  # Maximum sequence length to cap computational cost
INITIAL_TRAINING_DAYS = 783  # Number of days to use for initial training only
KELLY_FRACTION = 0.1
SECTOR_CONFIDENCE_THRESHOLD = 0.30
RETRAIN_INTERVAL = 200
MAX_DAY_GAP = 5  # Maximum allowed gap in trading days (to account for weekends/holidays)

# H=1 exit-monitor predictions (store probabilities only)
H_EXIT = 1
L_H1 = 12
MIN_SEQ = 50



In [ ]:
import random


def set_global_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seeds(42)

#### Define Horizon Target
For a horizon H, compute both the direction and H-day return off the same base day t

In [ ]:
def add_horizon_targets(df: pd.DataFrame, H: int, price_col='close') -> pd.DataFrame:
    df = df.sort_values(['ticker','date']).copy()
    df[f'ret_{H}d']    = df.groupby('ticker')[price_col].shift(-H) / df[price_col] - 1.0
    df[f'target_{H}d'] = (df[f'ret_{H}d'] > 0).astype(int)
    return df


#### Create Target Variable
Build the binary classification target per row.

In [ ]:
def create_target_variable(df: pd.DataFrame) -> pd.DataFrame:

    print("Creating target variable...")
    df = df.sort_values(by=['ticker', 'date']).copy()
    df['next_day_close'] = df.groupby('ticker')['close'].shift(-1)
    df['target'] = (df['next_day_close'] > df['close']).astype(int)
    df.dropna(subset=['next_day_close'], inplace=True)
    df['target'] = df['target'].astype(int)
    print("Target variable created.")
    return df

##### PyTorch Helpers

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden1=128, hidden2=64, fc=32, dropout=0.3, inter_dropout=0.1, use_layernorm=False):
        super().__init__()
        self.lstm1 = nn.LSTM(
            input_size=n_features, hidden_size=hidden1,
            num_layers=1, batch_first=True, bidirectional=False, dropout=0.1
        )
        self.inter_drop = nn.Dropout(p=inter_dropout)  # proxy for recurrent_dropout
        self.lstm2 = nn.LSTM(
            input_size=hidden1, hidden_size=hidden2,
            num_layers=1, batch_first=True, bidirectional=False, dropout=0.1
        )

        # Normalization after temporal pooling
        self.use_layernorm = bool(use_layernorm)
        if self.use_layernorm:
            self.norm = nn.LayerNorm(normalized_shape=hidden2)
        else:
            self.norm = nn.BatchNorm1d(num_features=hidden2)

        # Head
        self.fc1 = nn.Linear(hidden2, fc)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(p=dropout)
        self.fc_out = nn.Linear(fc, 1)  # logits

    def forward(self, x):
        # x: (B, T, F)
        out1, _ = self.lstm1(x)     # (B, T, hidden1)
        out1 = self.inter_drop(out1)
        out2, _ = self.lstm2(out1)  # (B, T, hidden2)

        last = out2[:, -1, :]       # (B, hidden2)
        if isinstance(self.norm, nn.BatchNorm1d):
            last = self.norm(last)  # BN expects (B, C)
        else:
            last = self.norm(last)  # LN expects (B, C)

        z = self.fc1(last)
        z = self.relu(z)
        z = self.drop(z)
        logits = self.fc_out(z).squeeze(-1)  # (B,)
        return logits

class EarlyStopper:
    def __init__(self, patience=15, mode='min'):
        self.patience = patience
        self.counter = 0
        self.best_metric = None
        self.best_state_dict = None
        self.mode = mode  # 'min' for val_loss
    def step(self, metric, model):
        if self.best_metric is None:
            improved = True
        else:
            improved = (metric < self.best_metric) if self.mode == 'min' else (metric > self.best_metric)
        if improved:
            self.best_metric = metric
            self.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
        return improved

@torch.no_grad()
def _evaluate(model, loader, device, loss_fn):
    model.eval()
    total_loss = 0.0
    all_logits = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        total_loss += loss.item() * xb.size(0)
        all_logits.append(logits.detach().cpu())
    avg_loss = total_loss / len(loader.dataset)
    logits = torch.cat(all_logits, dim=0).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))  # sigmoid
    return avg_loss, probs

### Load Data

In [ ]:
# master_df = pd.read_parquet('stocknet-dataset/master_df.parquet')
master_df = pd.read_parquet('stocknet-dataset/master_df_with_sector_and_bimeta_features.parquet')

In [ ]:
master_df.drop(columns=['text','adj close','sentiment','emotion_anger','emotion_disgust','emotion_fear','emotion_joy','emotion_neutral','emotion_sadness','emotion_surprize'], inplace=True)
# Exclude ticker with missing meta-features across all rows
master_df = master_df[master_df['ticker'] != 'GMRE']

columns_to_check = ['EMA_12', 'EMA_26', 'EMA_50', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9',
    'RSI_14', 'ATRr_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3',
    'BB_upper', 'BB_middle', 'BB_lower', 'OBV',
    
    'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d', 
    # 'sector_ret_60d',
    'sector_vol_20d', 'sector_dispersion_1d',
    'sector_rel_strength',
    
    # 'EMA_12_sector', 'EMA_26_sector', 'EMA_50_sector', 'MACD_12_26_9_sector', 'MACDh_12_26_9_sector', 
    # 'MACDs_12_26_9_sector', 'RSI_14_sector', 'sector_BB_upper', 'sector_BB_middle','sector_BB_lower',
    
    'lstm_prob_up_1d', 'lstm_brier_20', 'lstm_logloss_20', 'lstm_acc_20',
    
    'gru_pred_ret_1d', 'gru_abs_err_lag1', 'gru_mae_20', 'gru_rmse_20', 'gru_dir_acc_20',
                    ]

master_df = master_df.dropna(subset=columns_to_check)

master_df.reset_index(drop=True, inplace=True)

display(master_df)

,date,open,high,low,close,volume,ticker,stance_positive,stance_negative,sector,...,gru_rmse_20,gru_dir_acc_20,lstm_prob_up_1d,lstm_logit_up_1d,lstm_brier,lstm_logloss,lstm_correct,lstm_brier_20,lstm_logloss_20,lstm_acc_20
0,2014-03-04,75.857140,76.091431,75.395714,75.891426,64785000.0,AAPL,17.0,17.0,Consumer Goods,...,0.047611,0.600000,0.743493,1.064202,0.552781,1.360598,0.0,0.265279,0.723950,0.400000
1,2014-03-05,75.845711,76.392860,75.589996,76.051430,50015700.0,AAPL,13.0,8.0,Consumer Goods,...,0.045507,0.666667,0.728888,0.988988,0.073502,0.316235,1.0,0.313196,0.830058,0.333333
2,2014-03-06,76.112854,76.348572,75.442856,75.821426,46372200.0,AAPL,10.0,4.0,Consumer Goods,...,0.047723,0.571429,0.722673,0.957758,0.076910,0.324799,1.0,0.278954,0.756655,0.428571
3,2014-03-07,75.870003,75.997147,75.150002,75.777145,55182400.0,AAPL,23.0,5.0,Consumer Goods,...,0.049173,0.500000,0.686132,0.782095,0.470776,1.158781,0.0,0.253699,0.702673,0.500000
4,2014-03-10,75.480003,76.190002,75.477142,75.845711,44646000.0,AAPL,17.0,3.0,Consumer Goods,...,0.051356,0.555556,0.703303,0.863078,0.494636,1.215045,0.0,0.277818,0.753352,0.444444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75662,2017-08-25,76.559998,77.129997,76.430000,76.720001,6844900.0,XOM,0.0,0.0,Basic Matierials,...,0.024094,0.550000,0.518448,0.073827,0.231892,0.656915,1.0,0.253057,0.699271,0.450000
75663,2017-08-28,76.900002,76.940002,76.260002,76.470001,8229700.0,XOM,0.0,0.0,Basic Matierials,...,0.021807,0.550000,0.514868,0.059490,0.235353,0.663845,1.0,0.253938,0.701038,0.450000
75664,2017-08-29,76.209999,76.489998,76.080002,76.449997,7060400.0,XOM,0.0,0.0,Basic Matierials,...,0.020675,0.550000,0.523837,0.095419,0.226731,0.646575,1.0,0.255041,0.703251,0.450000
75665,2017-08-30,76.239998,76.449997,76.059998,76.099998,8218000.0,XOM,0.0,0.0,Basic Matierials,...,0.018285,0.550000,0.525389,0.101644,0.276034,0.745260,0.0,0.256353,0.705892,0.450000


In [ ]:
feature_columns = ['open','high','low', 'volume',
                   
                   'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_volume_mean',
    
                    # 'stance_positive','stance_negative'
                ]

new_indicator_columns = [
    'EMA_12', 'EMA_26', 'EMA_50', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9',
    'RSI_14', 'ATRr_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3',
    'BB_upper', 'BB_middle', 'BB_lower', 'OBV',
    
    'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d', 
    # 'sector_ret_60d',
    'sector_vol_20d', 'sector_dispersion_1d',
    'sector_rel_strength',
    
    # 'EMA_12_sector', 'EMA_26_sector', 'EMA_50_sector', 'MACD_12_26_9_sector', 'MACDh_12_26_9_sector', 
    # 'MACDs_12_26_9_sector', 'RSI_14_sector', 'sector_BB_upper', 'sector_BB_middle','sector_BB_lower'
    
    'lstm_prob_up_1d', 
    'lstm_brier_20', 'lstm_logloss_20', 'lstm_acc_20',
    
    # 'gru_pred_ret_1d', 
    # 'gru_abs_err_lag1', 'gru_mae_20', 'gru_rmse_20', 'gru_dir_acc_20',
]

feature_columns.extend(new_indicator_columns)
print(f"Final feature columns: {feature_columns}")
print(f"Total number of features: {len(feature_columns)}")

Final feature columns: ['open', 'high', 'low', 'volume', 'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_volume_mean', 'EMA_12', 'EMA_26', 'EMA_50', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'RSI_14', 'ATRr_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3', 'BB_upper', 'BB_middle', 'BB_lower', 'OBV', 'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d', 'sector_vol_20d', 'sector_dispersion_1d', 'sector_rel_strength', 'lstm_prob_up_1d', 'lstm_brier_20', 'lstm_logloss_20', 'lstm_acc_20']
Total number of features: 32


In [ ]:
master_df.reset_index(drop=True, inplace=True)
master_df

,date,open,high,low,close,volume,ticker,stance_positive,stance_negative,sector,...,gru_rmse_20,gru_dir_acc_20,lstm_prob_up_1d,lstm_logit_up_1d,lstm_brier,lstm_logloss,lstm_correct,lstm_brier_20,lstm_logloss_20,lstm_acc_20
0,2014-03-04,75.857140,76.091431,75.395714,75.891426,64785000.0,AAPL,17.0,17.0,Consumer Goods,...,0.047611,0.600000,0.743493,1.064202,0.552781,1.360598,0.0,0.265279,0.723950,0.400000
1,2014-03-05,75.845711,76.392860,75.589996,76.051430,50015700.0,AAPL,13.0,8.0,Consumer Goods,...,0.045507,0.666667,0.728888,0.988988,0.073502,0.316235,1.0,0.313196,0.830058,0.333333
2,2014-03-06,76.112854,76.348572,75.442856,75.821426,46372200.0,AAPL,10.0,4.0,Consumer Goods,...,0.047723,0.571429,0.722673,0.957758,0.076910,0.324799,1.0,0.278954,0.756655,0.428571
3,2014-03-07,75.870003,75.997147,75.150002,75.777145,55182400.0,AAPL,23.0,5.0,Consumer Goods,...,0.049173,0.500000,0.686132,0.782095,0.470776,1.158781,0.0,0.253699,0.702673,0.500000
4,2014-03-10,75.480003,76.190002,75.477142,75.845711,44646000.0,AAPL,17.0,3.0,Consumer Goods,...,0.051356,0.555556,0.703303,0.863078,0.494636,1.215045,0.0,0.277818,0.753352,0.444444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75662,2017-08-25,76.559998,77.129997,76.430000,76.720001,6844900.0,XOM,0.0,0.0,Basic Matierials,...,0.024094,0.550000,0.518448,0.073827,0.231892,0.656915,1.0,0.253057,0.699271,0.450000
75663,2017-08-28,76.900002,76.940002,76.260002,76.470001,8229700.0,XOM,0.0,0.0,Basic Matierials,...,0.021807,0.550000,0.514868,0.059490,0.235353,0.663845,1.0,0.253938,0.701038,0.450000
75664,2017-08-29,76.209999,76.489998,76.080002,76.449997,7060400.0,XOM,0.0,0.0,Basic Matierials,...,0.020675,0.550000,0.523837,0.095419,0.226731,0.646575,1.0,0.255041,0.703251,0.450000
75665,2017-08-30,76.239998,76.449997,76.059998,76.099998,8218000.0,XOM,0.0,0.0,Basic Matierials,...,0.018285,0.550000,0.525389,0.101644,0.276034,0.745260,0.0,0.256353,0.705892,0.450000


In [ ]:
folder_path = 'trained_models/'
if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
else:
    print(f"Folder not found: {folder_path}")

print(f"Contents of {folder_path} after deletion attempt:")
if os.path.exists(folder_path):
    print(os.listdir(folder_path))
else:
    print("Folder does not exist.")

Contents of trained_models/ after deletion attempt:
[]


In [ ]:
FEATURES_H1_EXIT = feature_columns

h1_df = add_horizon_targets(master_df.copy(), H=H_EXIT, price_col='close')
ret_col = f'ret_{H_EXIT}d'
tgt_col = f'target_{H_EXIT}d'

h1_df = h1_df.dropna(subset=FEATURES_H1_EXIT + [ret_col, tgt_col])
unique_dates = sorted(h1_df['date'].unique())
train_cutoff_date = unique_dates[INITIAL_TRAINING_DAYS - 1]
last_100_dates = set(unique_dates[-100:])
print(f"Train cutoff: {train_cutoff_date}")
print(f"Last 100 window: {min(last_100_dates)} -> {max(last_100_dates)}")

def build_sequences_idx(df, feature_cols, target_col, seq_len):
    X_vals = df[feature_cols].values
    y_vals = df[target_col].values
    X_list, y_list, idx_list = [], [], []
    for i in range(seq_len - 1, len(df)):
        window = X_vals[i - seq_len + 1:i + 1]
        if np.isnan(window).any():
            continue
        if pd.isna(y_vals[i]):
            continue
        X_list.append(window)
        y_list.append(y_vals[i])
        idx_list.append(i)
    return np.array(X_list), np.array(y_list), idx_list

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pred_rows = []

for ticker, df_tkr in h1_df.groupby('ticker'):
    df_tkr = df_tkr.sort_values('date').reset_index(drop=True)
    df_tkr = df_tkr.dropna(subset=FEATURES_H1_EXIT + [ret_col, tgt_col])
    print(f"[{ticker}] rows after dropna: {len(df_tkr)}")
    if len(df_tkr) < (L_H1 + 10):
        print(f"[{ticker}] skipped: too few rows")
        continue

    X_all, y_all, idx_list = build_sequences_idx(df_tkr, FEATURES_H1_EXIT, tgt_col, L_H1)
    print(f"[{ticker}] sequences built: {len(X_all)}")
    if len(X_all) < MIN_SEQ:
        print(f"[{ticker}] skipped: too few sequences")
        continue

    idx_arr = np.array(idx_list)
    idx_to_seq = {idx_list[i]: i for i in range(len(idx_list))}

    end_dates = df_tkr.iloc[idx_list]['date']
    pred_end_idxs = end_dates[(end_dates > train_cutoff_date) & (end_dates.isin(last_100_dates))].index.tolist()
    print(f"[{ticker}] prediction indices: {len(pred_end_idxs)}")
    if not pred_end_idxs:
        print(f"[{ticker}] no post-cutoff indices; date range: {df_tkr['date'].min()} -> {df_tkr['date'].max()}")
        continue
    pred_end_idxs = sorted(pred_end_idxs)

    pos = 0
    while pos < len(pred_end_idxs):
        block_start_idx = pred_end_idxs[pos]
        if block_start_idx < 2:
            pos += RETRAIN_INTERVAL
            continue
        train_end_idx = block_start_idx - 2  # avoid label leakage at t-1
        train_mask = idx_arr <= train_end_idx
        if train_mask.sum() < MIN_SEQ:
            pos += RETRAIN_INTERVAL
            continue

        X_train_full = X_all[train_mask]
        y_train_full = y_all[train_mask]
        split = max(int(len(X_train_full) * 0.8), 1)
        if split >= len(X_train_full):
            pos += RETRAIN_INTERVAL
            continue
        X_tr, y_tr = X_train_full[:split], y_train_full[:split]
        X_val, y_val = X_train_full[split:], y_train_full[split:]

        scaler = MinMaxScaler()
        F = X_tr.shape[-1]
        X_tr_s = scaler.fit_transform(X_tr.reshape(-1, F)).reshape(X_tr.shape)
        X_val_s = scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape)

        train_ds = SequenceDataset(X_tr_s, y_tr)
        val_ds   = SequenceDataset(X_val_s, y_val)

        train_bs = min(32, len(train_ds))
        if train_bs < 2:
            pos += RETRAIN_INTERVAL
            continue
        if len(train_ds) % train_bs == 1 and train_bs > 2:
            train_bs -= 1
        val_bs = min(32, len(val_ds))

        train_loader = DataLoader(train_ds, batch_size=train_bs, shuffle=True, drop_last=False)
        val_loader   = DataLoader(val_ds,   batch_size=val_bs, shuffle=False, drop_last=False)

        model = LSTMClassifier(n_features=F, dropout=0.3, inter_dropout=0.1).to(device)
        loss_fn = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-7)
        early = EarlyStopper(patience=15, mode='min')

        for epoch in range(100):
            model.train()
            total_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_loss += loss.item() * xb.size(0)

            val_loss, _ = _evaluate(model, val_loader, device, loss_fn)
            scheduler.step(val_loss)
            _ = early.step(val_loss, model)
            if early.counter >= early.patience:
                break

        if early.best_state_dict is not None:
            model.load_state_dict(early.best_state_dict)

        _, val_probs = _evaluate(model, val_loader, device, loss_fn)
        if len(val_probs) == 0:
            pos += RETRAIN_INTERVAL
            continue
        pred_min, pred_max = val_probs.min(), val_probs.max()
        buffer = 0.05 * (pred_max - pred_min)
        y_min_dynamic = max(0.0, pred_min - buffer)
        y_max_dynamic = min(1.0, pred_max + buffer)
        calibrator = IsotonicRegression(y_min=y_min_dynamic, y_max=y_max_dynamic, out_of_bounds='clip')
        calibrator.fit(val_probs, y_val.astype(float))

        block_indices = pred_end_idxs[pos:pos + RETRAIN_INTERVAL]
        X_pred = []
        pred_dates = []
        for idx in block_indices:
            seq_idx = idx_to_seq.get(idx)
            if seq_idx is None:
                continue
            X_pred.append(X_all[seq_idx])
            pred_dates.append(df_tkr.iloc[idx]['date'])

        if X_pred:
            X_pred = np.stack(X_pred)
            X_pred_s = scaler.transform(X_pred.reshape(-1, F)).reshape(X_pred.shape)
            with torch.no_grad():
                xb = torch.from_numpy(X_pred_s.astype(np.float32)).to(device)
                logits = model(xb)
                raw_probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
            cal_probs = calibrator.predict(raw_probs)
            for d, p in zip(pred_dates, cal_probs):
                pred_rows.append({
                    'date': pd.to_datetime(d),
                    'ticker': ticker,
                    'p_up_1d': float(np.clip(p, 0.0, 1.0)),
                })

        pos += RETRAIN_INTERVAL

h1_exit_df = pd.DataFrame(pred_rows)
if h1_exit_df.empty:
    print('h1_exit_df is empty; no predictions generated.')
    h1_exit_df = pd.DataFrame(columns=['date','ticker','p_up_1d'])
else:
    h1_exit_df = h1_exit_df.drop_duplicates(subset=['date', 'ticker'])
    h1_exit_df = h1_exit_df.sort_values(['date', 'ticker']).reset_index(drop=True)

# keep only last 100 dates for simulator
before_rows = len(h1_exit_df)
h1_exit_df = h1_exit_df[h1_exit_df['date'].isin(last_100_dates)].copy()
after_rows = len(h1_exit_df)
print(f"Filtered to last 100 dates: {before_rows} -> {after_rows} rows")
h1_exit_df = h1_exit_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print('h1_exit_df shape:', h1_exit_df.shape)
if not h1_exit_df.empty:
    print('p_up_1d range:', h1_exit_df['p_up_1d'].min(), h1_exit_df['p_up_1d'].max())
    print('unique dates:', h1_exit_df['date'].nunique())

out_path = 'stocknet-dataset/h1_exit_df.parquet'
h1_exit_df.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')


Train cutoff: 2017-04-10 00:00:00
Last 100 window: 2017-04-10 00:00:00 -> 2017-08-30 00:00:00
[AAPL] rows after dropna: 882
[AAPL] sequences built: 871
[AAPL] prediction indices: 99
[ABB] rows after dropna: 882
[ABB] sequences built: 871
[ABB] prediction indices: 99
[ABBV] rows after dropna: 820
[ABBV] sequences built: 809
[ABBV] prediction indices: 99
[AEP] rows after dropna: 882
[AEP] sequences built: 871
[AEP] prediction indices: 99
[AGFS] rows after dropna: 344
[AGFS] sequences built: 333
[AGFS] prediction indices: 99
[AMGN] rows after dropna: 882
[AMGN] sequences built: 871
[AMGN] prediction indices: 99
[AMZN] rows after dropna: 882
[AMZN] sequences built: 871
[AMZN] prediction indices: 99
[BA] rows after dropna: 882
[BA] sequences built: 871
[BA] prediction indices: 99
[BABA] rows after dropna: 388
[BABA] sequences built: 377
[BABA] prediction indices: 99
[BAC] rows after dropna: 882
[BAC] sequences built: 871
[BAC] prediction indices: 99
[BBL] rows after dropna: 882
[BBL] sequen

In [ ]:
# Sanity-check coverage and build lookup index
h1_exit_df = pd.read_parquet('stocknet-dataset/h1_exit_df.parquet')
h1_exit_df['date'] = pd.to_datetime(h1_exit_df['date'])

dup_count = h1_exit_df.duplicated(subset=['date','ticker']).sum()
print(f'Duplicate (date,ticker) rows: {dup_count}')
if dup_count > 0:
    h1_exit_df = h1_exit_df.drop_duplicates(subset=['date','ticker'])

unique_dates = h1_exit_df['date'].nunique()
unique_tickers = h1_exit_df['ticker'].nunique()
min_date = h1_exit_df['date'].min()
max_date = h1_exit_df['date'].max()
avg_tickers_per_date = h1_exit_df.groupby('date')['ticker'].nunique().mean()

print(f'Unique dates: {unique_dates}')
print(f'Unique tickers: {unique_tickers}')
print(f'Date range: {min_date} -> {max_date}')
print(f'Avg tickers per date: {avg_tickers_per_date:.2f}')

h1_exit_lookup = h1_exit_df.set_index(['date','ticker'])
print('Lookup index built:', h1_exit_lookup.index.is_unique)


Duplicate (date,ticker) rows: 0
Unique dates: 99
Unique tickers: 87
Date range: 2017-04-11 00:00:00 -> 2017-08-30 00:00:00
Avg tickers per date: 87.00
Lookup index built: True


In [ ]:
# Load the two parquet files
df_old = pd.read_parquet('stocknet-dataset/h1_exit_df_old.parquet')
df_new = pd.read_parquet('stocknet-dataset/h1_exit_df.parquet')

# Ensure the 'date' column is in datetime format for both dataframes
df_old['date'] = pd.to_datetime(df_old['date'])
df_new['date'] = pd.to_datetime(df_new['date'])

# Merge the two dataframes to find differences
comparison_df = df_old.merge(df_new, on=['date', 'ticker'], how='outer', suffixes=('_old', '_new'), indicator=True)

# Highlight rows that are different
differences = comparison_df[comparison_df['_merge'] != 'both']

# Display the differences
print("Differences between the two parquet files:")
print(differences)

# Save the differences to a new file for further analysis
differences.to_csv('differences_between_parquets.csv', index=False)
print("Differences saved to 'differences_between_parquets.csv'")